# Cold-start demand prediction: results

Reads the files written by `python -m coldstart.run` in `../results` and shows the
evaluation: split sizes, pooled metrics with product-level bootstrap intervals, paired
differences between models, and how predictions compare with actual sales.

Run `python -m coldstart.run --raw-dir <H&M folder> --out results` first (see the README).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from coldstart.evaluate import format_deltas, format_pooled

RESULTS = Path("../results")
splits = pd.read_csv(RESULTS / "splits.csv")
pooled = pd.read_csv(RESULTS / "metrics_pooled.csv")
by_origin = pd.read_csv(RESULTS / "metrics_by_origin.csv")
deltas = pd.read_csv(RESULTS / "paired_deltas.csv")
preds = pd.read_csv(RESULTS / "predictions.csv.gz")
splits

## Pooled metrics

WAPE and relative bias over all test windows, with 95% intervals from a bootstrap that
resamples whole products (colour variants of a product have correlated sales).

In [ ]:
format_pooled(pooled)

## Differences between models

Paired bootstrap of the WAPE difference. A negative delta means the first model is better;
the last column says whether the interval excludes zero.

In [ ]:
format_deltas(deltas)

## Stability across forecast origins

If a model only wins on one origin, the pooled number hides it.

In [ ]:
wape_by_origin = by_origin.pivot(index="model", columns="origin", values="wape")
wape_by_origin.loc[pooled["model"]].round(3)

In [ ]:
ax = wape_by_origin.loc[pooled["model"]].plot.barh(figsize=(7, 4))
ax.invert_yaxis()
ax.set(xlabel="WAPE (lower is better)", ylabel="", title="WAPE by forecast origin")
plt.tight_layout()
plt.show()

## Predictions against actual sales

Sales are very right-skewed, so both axes are on a log scale. A model trained with an L1 loss
predicts the median, which is why the points sit below the diagonal for the best sellers.

In [ ]:
best = pooled.sort_values("wape").iloc[0]["model"]
g = preds[preds["model"] == best]
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(np.log1p(g["pred"]), np.log1p(g["units"]), s=3, alpha=0.2)
lim = [0, max(np.log1p(g["units"]).max(), np.log1p(g["pred"]).max())]
ax.plot(lim, lim, color="k", lw=1)
ax.set(xlabel="log(1 + predicted units)", ylabel="log(1 + actual units)", title=best)
plt.tight_layout()
plt.show()